In [1]:
# ═══ CORE ═══════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd

# ═══ VISUALIZATION ════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mtick
from matplotlib.patches import Patch
import seaborn as sns

# ═══ STATS ════════════════════════════════════════════════════════════════════
from scipy import stats
from scipy.stats import (
    skew, kurtosis, describe,
    shapiro, normaltest, anderson, kstest,
    pearsonr, spearmanr, kendalltau, pointbiserialr,
    chi2_contingency, fisher_exact,
    ttest_ind, mannwhitneyu, f_oneway, kruskal,
    ks_2samp, levene, bartlett,
    boxcox, yeojohnson, zscore, iqr
)
from scipy.cluster import hierarchy as sch

# ═══ SKLEARN (pre-model only) ════════════════════════════════════════════════
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer, KNNImputer

# ═══ DISPLAY CONFIG ═══════════════════════════════════════════════════════════
import warnings; warnings.filterwarnings('ignore')
from IPython.display import display, HTML

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 60)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# ═══ PLOT THEME ═══════════════════════════════════════════════════════════════
PALETTE   = sns.color_palette("tab10")
DIV_PAL   = "coolwarm"
SEQ_PAL   = "Blues"
BG        = "#F8F8F8"

sns.set_theme(style="whitegrid", palette="tab10",
              rc={"figure.facecolor": BG, "axes.facecolor": BG,
                  "axes.spines.top": False, "axes.spines.right": False,
                  "grid.color": "#E0E0E0", "grid.linewidth": 0.6})
plt.rcParams.update({"figure.dpi": 120, "font.size": 11,
                      "axes.titlesize": 12, "axes.titleweight": "bold"})

# ═══ PROJECT CONSTANTS ════════════════════════════════════════════════════════
TARGET      = "target"         # ← your target column
TASK_TYPE   = "classification" # ← "classification" | "regression"
ID_COL      = "id"             # ← ID column or None
RANDOM_SEED = 42
N_SAMPLE    = 5_000            # sample size for heavy plots on large datasets

print("✓ Setup complete")


✓ Setup complete


In [2]:
# ═══ MASTER HELPER FUNCTIONS (do not edit mid-project) ═══════════════════════

# ─── Audit helpers ────────────────────────────────────────────────────────────
def df_audit(df):
    """Full schema audit: dtype, null %, unique count, sample, memory."""
    d = pd.DataFrame({
        "dtype":    df.dtypes,
        "n_null":   df.isnull().sum(),
        "pct_null": (df.isnull().mean() * 100).round(2),
        "n_unique": df.nunique(),
        "pct_unique": (df.nunique() / len(df) * 100).round(2),
        "sample_val": [df[c].dropna().iloc[0] if df[c].notna().any() else None
                       for c in df.columns],
        "mem_kb":   (df.memory_usage(deep=True) / 1024).round(2).values[1:]
    })
    d.index.name = "feature"
    return d.sort_values("pct_null", ascending=False)

def null_heatmap(df, figsize=(14, 5)):
    """Null map heatmap. Yellow = missing."""
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(df.isnull(), yticklabels=False, cbar=False,
                cmap="viridis", ax=ax)
    ax.set_title("Missing value map  (yellow = null)", fontsize=13)
    plt.tight_layout(); plt.show()

def null_bar(df, figsize=(12, 4)):
    """Bar chart of null % per column, sorted desc."""
    null_pct = df.isnull().mean().sort_values(ascending=False)
    null_pct = null_pct[null_pct > 0]
    if null_pct.empty:
        print("No nulls."); return
    fig, ax = plt.subplots(figsize=figsize)
    colors = ["#e53935" if v > 0.4 else "#fb8c00" if v > 0.1 else "#1e88e5"
              for v in null_pct.values]
    ax.barh(null_pct.index[::-1], null_pct.values[::-1] * 100, color=colors[::-1])
    ax.set_xlabel("Missing %")
    ax.set_title("Missing values per feature")
    ax.axvline(40, color="red",    linestyle="--", linewidth=1, alpha=0.7, label="40%")
    ax.axvline(10, color="orange", linestyle="--", linewidth=1, alpha=0.7, label="10%")
    ax.legend(title="Threshold")
    plt.tight_layout(); plt.show()

# ─── Distribution helpers ──────────────────────────────────────────────────────
def dist_box_qq(series, title=None):
    """3-panel: histogram+KDE / boxplot / Q-Q plot. For any numeric series."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    s = series.dropna()
    title = title or series.name
    # hist + KDE
    sns.histplot(s, kde=True, ax=axes[0], color=PALETTE[0], edgecolor="white")
    axes[0].axvline(s.mean(),   color="red",    linestyle="--", label=f"mean={s.mean():.2f}")
    axes[0].axvline(s.median(), color="green",  linestyle="--", label=f"med={s.median():.2f}")
    axes[0].legend(fontsize=8); axes[0].set_title(f"Distribution: {title}")
    # boxplot
    axes[1].boxplot(s, vert=True, patch_artist=True,
                    boxprops=dict(facecolor=PALETTE[1], alpha=0.6),
                    medianprops=dict(color="black", linewidth=2))
    axes[1].set_title(f"Boxplot: {title}")
    # Q-Q
    stats.probplot(s, dist="norm", plot=axes[2])
    axes[2].set_title(f"Q-Q plot: {title}")
    plt.suptitle(f"Skew={skew(s):.3f}  Kurt={kurtosis(s):.3f}  n={len(s)}", y=1.02)
    plt.tight_layout(); plt.show()

# ─── Bivariate helpers ─────────────────────────────────────────────────────────
def num_vs_target(df, col, target=TARGET, task=TASK_TYPE):
    """3-panel numeric feature vs target view."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    s = df[col].dropna()
    sns.histplot(df[col].dropna(), kde=True, ax=axes[0], color=PALETTE[0])
    axes[0].set_title(f"Distribution: {col}")
    if task == "regression":
        axes[1].scatter(df[col], df[target], alpha=0.3, s=10, color=PALETTE[0])
        m, b = np.polyfit(df[[col, target]].dropna()[col],
                          df[[col, target]].dropna()[target], 1)
        xr = np.linspace(df[col].min(), df[col].max(), 100)
        axes[1].plot(xr, m*xr+b, color="red", lw=1.5)
        axes[1].set_xlabel(col); axes[1].set_ylabel(target)
        axes[1].set_title(f"{col} vs {target}")
        r, p = pearsonr(df[[col, target]].dropna()[col], df[[col, target]].dropna()[target])
        axes[1].text(0.05, 0.95, f"r={r:.3f} p={p:.3f}", transform=axes[1].transAxes,
                     va='top', fontsize=9, color="navy")
    else:
        sns.boxplot(data=df, x=target, y=col, ax=axes[1], palette="tab10")
        axes[1].set_title(f"{col} by {target}")
    sns.boxplot(y=df[col].dropna(), ax=axes[2], color=PALETTE[1])
    axes[2].set_title(f"Boxplot: {col}")
    plt.tight_layout(); plt.show()

def cat_vs_target(df, col, target=TARGET, task=TASK_TYPE, max_cats=20):
    """2-panel categorical feature vs target view."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    vc = df[col].value_counts().head(max_cats)
    sns.barplot(y=vc.index.astype(str), x=vc.values, ax=axes[0], palette="tab10")
    axes[0].set_title(f"Value counts: {col}")
    if task == "classification":
        ct = pd.crosstab(df[col], df[target], normalize="index") * 100
        ct.head(max_cats).plot(kind="barh", stacked=True, ax=axes[1], colormap="Set2")
        axes[1].set_title(f"{col} → class rate (%)")
    else:
        order = df.groupby(col)[target].median().sort_values().index[:max_cats]
        sns.boxplot(data=df[df[col].isin(order)], x=target, y=col,
                    order=order, ax=axes[1], palette="tab10")
        axes[1].set_title(f"{target} by {col}")
    plt.tight_layout(); plt.show()

# ─── Statistical helpers ───────────────────────────────────────────────────────
def cramers_v(x, y):
    """Cramér's V association between two categoricals."""
    ct = pd.crosstab(x, y)
    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.values.sum()
    r, k = ct.shape
    return round(np.sqrt(chi2 / (n * (min(k, r) - 1))), 4)

def iqr_outlier_report(df, cols=None):
    """IQR-based outlier count per numeric column."""
    cols = cols or df.select_dtypes(include=np.number).columns.tolist()
    rows = []
    for c in cols:
        q1, q3 = df[c].quantile(0.25), df[c].quantile(0.75)
        iqr_val = q3 - q1
        lo, hi = q1 - 1.5*iqr_val, q3 + 1.5*iqr_val
        n = ((df[c] < lo) | (df[c] > hi)).sum()
        rows.append({"feature":c, "Q1":round(q1,3), "Q3":round(q3,3),
                     "IQR":round(iqr_val,3), "lower_fence":round(lo,3),
                     "upper_fence":round(hi,3), "n_outliers":n,
                     "pct_outliers":round(n/len(df)*100, 2)})
    return pd.DataFrame(rows).sort_values("n_outliers", ascending=False)

def normality_suite(series, alpha=0.05):
    """Run Shapiro-Wilk, D'Agostino-Pearson, Anderson-Darling, KS test."""
    s = series.dropna()
    results = {}
    # Shapiro-Wilk (best for n<5000)
    if len(s) <= 5000:
        stat, p = shapiro(s)
        results["Shapiro-Wilk"] = {"stat":round(stat,5), "p":round(p,5),
                                    "normal": p > alpha}
    # D'Agostino-Pearson
    stat, p = normaltest(s)
    results["D'Agostino-Pearson"] = {"stat":round(stat,5), "p":round(p,5),
                                      "normal": p > alpha}
    # Anderson-Darling
    ad = anderson(s, dist='norm')
    results["Anderson-Darling"] = {"stat":round(ad.statistic,5),
                                    "critical_5pct": ad.critical_values[2],
                                    "normal": ad.statistic < ad.critical_values[2]}
    # KS test vs fitted normal
    stat, p = kstest(s, 'norm', args=(s.mean(), s.std()))
    results["KS-test"] = {"stat":round(stat,5), "p":round(p,5),
                           "normal": p > alpha}
    return pd.DataFrame(results).T

print("✓ All helper functions loaded  |  helpers: df_audit · null_heatmap · null_bar")
print("   dist_box_qq · num_vs_target · cat_vs_target")
print("   cramers_v · iqr_outlier_report · normality_suite")


✓ All helper functions loaded  |  helpers: df_audit · null_heatmap · null_bar
   dist_box_qq · num_vs_target · cat_vs_target
   cramers_v · iqr_outlier_report · normality_suite
